# SME-GPT — Llama 3 Fine-tuning Notebook

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Upload `sme_gpt_train.jsonl` and `sme_gpt_eval.jsonl` to the Files panel on the left

**What this does:**
- Fine-tunes Llama 3.1 8B Instruct with QLoRA (4-bit, fits in T4 15 GB VRAM)
- Trains on your SME document extraction + correction examples
- Exports a GGUF Q4_K_M model you load into local Ollama

In [ ]:
%%capture
import os
# Fix the protobuf conflict (root cause of the error)
os.system('pip install -q --upgrade "protobuf>=5.26.1"')
# Remove broken old install
os.system('pip uninstall -y unsloth unsloth_zoo 2>/dev/null')
# Install latest from GitHub (not pinned version)
os.system('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
# Training dependencies
os.system('pip install -q --upgrade "trl>=0.15.0" peft accelerate bitsandbytes datasets transformers')
print('Done. Now do: Runtime > Restart runtime')


In [ ]:
import torch
from unsloth import FastLanguageModel
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Unsloth: OK')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Unsloth: OK


In [ ]:
# Cell 2 — Load base model with Unsloth (4-bit, fits T4 15 GB)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
DTYPE = None        # auto-detect
LOAD_IN_4BIT = True # QLoRA — halves VRAM usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Meta-Llama-3.1-8B-Instruct',
    max_seq_length = MAX_SEQ_LEN,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)
print('Base model loaded.')

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded.


In [ ]:
# Cell 3 — Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA rank — 16 is a good balance
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                       'gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
print('LoRA adapters attached.')

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters attached.


In [ ]:
# Cell 4 — Load and format the training data
from datasets import load_dataset

# Alpaca prompt template
ALPACA_TEMPLATE = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_example(examples):
    texts = []
    for instruction, inp, output in zip(
        examples['instruction'], examples['input'], examples['output']
    ):
        text = ALPACA_TEMPLATE.format(
            instruction=instruction, input=inp, output=output
        ) + EOS_TOKEN
        texts.append(text)
    return {'text': texts}

# Load the JSONL files you uploaded
train_dataset = load_dataset('json', data_files='/content/sme_gpt_train.jsonl', split='train')
eval_dataset  = load_dataset('json', data_files='/content/sme_gpt_eval.jsonl',  split='train')

train_dataset = train_dataset.map(format_example, batched=True)
eval_dataset  = eval_dataset.map(format_example,  batched=True)

print(f'Train: {len(train_dataset)} examples')
print(f'Eval:  {len(eval_dataset)} examples')
print('\nSample:')
print(train_dataset[0]['text'][:400])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Train: 55 examples
Eval:  6 examples

Sample:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a financial document data extraction engine for Sri Lankan SME businesses. You output ONLY a single valid JSON object — no markdown fences, no explanation, no text before or after the JSON. Start your response with {


In [ ]:
# Cell 5 — Train (~20-30 min on T4)
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    args = SFTConfig(
        # ── data ──
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LEN,
        dataset_num_proc            = 2,
        packing                     = False,
        padding_free                = False,
        # ── training ──
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        seed                        = 42,
        output_dir                  = '/content/sme_gpt_checkpoints',
        report_to                   = 'none',
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Training complete. Final loss: {trainer_stats.training_loss:.4f}')


/content/unsloth_compiled_cache/UnslothSFTTrainer.py:657: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  super().__init__(


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/55 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.723840,1.335265
2,1.132453,0.584272
3,0.351999,0.442675


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training complete. Final loss: 0.9345


In [ ]:
# Cell 6 — Quick test before exporting
FastLanguageModel.for_inference(model)

test_input = """Invoice No: INV-2024-001
Date: 15/01/2024
From: Colombo Traders
To: ABC Company
Item: Office Chair x2 @ LKR 8500 = LKR 17000
Total: LKR 17000"""

prompt = ALPACA_TEMPLATE.format(
    instruction="Extract structured JSON from this financial document OCR text.",
    input=test_input,
    output=''
)

inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.01, do_sample=True)
result = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print('Model output:')
print(result)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model output:
{
  "invoice_number": "INV-2024-001",
  "date": "15/01/2024",
  "from": "Colombo Traders",
  "to": "ABC Company",
  "items": [
    {
      "description": "Office Chair",
      "quantity": 2,
      "unit_price": 8500,
      "line_total": 17000
    }
  ],
  "document_total": 17000
}


In [ ]:
# Cell 7 — Export to GGUF Q4_K_M (Ollama-compatible, ~4.7 GB)
print('Exporting to GGUF...')
model.save_pretrained_gguf(
    'sme_gpt_llama3',
    tokenizer,
    quantization_method = 'q4_k_m',   # best size/quality balance
)
print('GGUF saved to /content/sme_gpt_llama3/')

# List the output files
import os
for f in os.listdir('/content/sme_gpt_llama3/'):
    size = os.path.getsize(f'/content/sme_gpt_llama3/{f}') / 1e9
    print(f'  {f}  ({size:.2f} GB)')

Exporting to GGUF...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 15073.87it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:20<00:00, 125.01s/it]


Unsloth: Merge process complete. Saved to `/content/sme_gpt_llama3`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


KeyboardInterrupt: 

In [ ]:
# Cell 8 — Download the GGUF to your computer
# Method A: Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/sme_gpt_llama3'
shutil.copytree('/content/sme_gpt_llama3', dest, dirs_exist_ok=True)
print(f'Copied to Google Drive: {dest}')
print('\nDownload the .gguf file from Google Drive to your local machine.')
print('Then run Phase 4 to load it into Ollama.')

Mounted at /content/drive
Copied to Google Drive: /content/drive/MyDrive/sme_gpt_llama3

Download the .gguf file from Google Drive to your local machine.
Then run Phase 4 to load it into Ollama.


In [ ]:
from google.colab import files
import glob
for f in glob.glob('/content/sme_gpt_llama3/*.gguf'):
    files.download(f)


In [ ]:
# Cell 8 — Download the GGUF to your computer
# Method A: Google Drive (recommended for large files)
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/sme_gpt_llama3_gguf'
shutil.copytree('/content/sme_gpt_llama3_gguf', dest, dirs_exist_ok=True)
print(f'Copied to Google Drive: {dest}')
print('\nDownload the .gguf file from Google Drive to your local machine.')
print('Then run Phase 4 to load it into Ollama.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to Google Drive: /content/drive/MyDrive/sme_gpt_llama3_gguf

Download the .gguf file from Google Drive to your local machine.
Then run Phase 4 to load it into Ollama.


In [ ]:
# Cell 8 (Alternative) — Direct browser download (only works for files < 2 GB)
# Use this if you don't want to use Google Drive
from google.colab import files
import glob

gguf_files = glob.glob('/content/sme_gpt_llama3/*.gguf')
for f in gguf_files:
    print(f'Downloading {f}...')
    files.download(f)

## After downloading the GGUF file

Run these commands on your local machine to register the model with Ollama:

```bash
# 1. Move the .gguf file to a known location
# e.g.  C:\models\sme-gpt-llama3.Q4_K_M.gguf

# 2. Create a Modelfile
# (the generate_modelfile.py script in backend/scripts/ does this automatically)
python backend/scripts/generate_modelfile.py C:\models\sme-gpt-llama3.Q4_K_M.gguf

# 3. Register with Ollama
ollama create sme-gpt-llama3 -f backend/training_data/Modelfile

# 4. Test it
ollama run sme-gpt-llama3

# 5. Update OLLAMA_MODEL in backend/.env
# OLLAMA_MODEL=sme-gpt-llama3
```